# Experiment 1: Automated Machine Learning Workflow
This standalone notebook implements a complete end-to-end automated data ingestion, exploratory data analysis (EDA), preprocessing, and feature selection pipeline.

In [1]:
import os
import matplotlib
matplotlib.use('Agg') # Strictly headless - non-interfering, zero GUI popups

def resolve_path(rel_path):
    """Dynamically resolves datasets whether run from repo root or Ex subfolder."""
    for prefix in ['', '../', '../../']:
        cand = os.path.join(prefix, rel_path)
        if os.path.exists(cand):
            return cand
    return rel_path

def resolve_out(rel_path):
    """Avoids nested directories if running from within Ex1."""
    if os.path.basename(os.getcwd()) == 'Ex1':
        if rel_path.startswith('Ex1/'):
            return rel_path[len('Ex1/'):]
    return rel_path

import pandas as pd
import numpy as np
import struct
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs(resolve_out('Ex1/plots'), exist_ok=True)

In [2]:
def smart_loader(data_source):
    if isinstance(data_source, dict):
        loaded_splits = {}
        for split_name, file_path in data_source.items():
            data, data_type = smart_loader(file_path)
            loaded_splits[split_name] = data
        return loaded_splits, f"{data_type}_split"

    resolved = resolve_path(data_source)
    if not os.path.exists(resolved):
        raise FileNotFoundError(f"File not found: {data_source} (checked {resolved})")

    _, ext = os.path.splitext(resolved)
    ext = ext.lower()
    basename = os.path.basename(resolved).lower()

    if ext in ['.csv', '.data']:
        if 'email' in basename or 'spam' in basename:
            return pd.read_csv(resolved), "nlp_vectorized"
        if 'iris' in basename:
            return pd.read_csv(resolved, header=None, names=['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class']), "tabular_multiclass"
        if 'diabetes' in basename:
            return pd.read_csv(resolved), "tabular_binary"
        if 'loan' in basename:
            df = pd.read_csv(resolved)
            df.columns = df.columns.str.strip()
            return df, "tabular_binary"
        return pd.read_csv(resolved), "tabular_generic"

    if ext in ['.idx3-ubyte', '.ubyte']:
        with open(resolved, 'rb') as f:
            magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
            images = np.fromfile(f, dtype=np.uint8)
            images = images[:500 * rows * cols].reshape(500, rows * cols)
        return pd.DataFrame(images), "image_raw"

    return pd.read_csv(resolved), "tabular_fallback"

In [3]:
def automated_eda(df, data_type):
    print(f"Dataset Shape: {df.shape}")
    print(f"Missing Values: {df.isnull().sum().sum()}")
    print("Summary Statistics:")
    display(df.describe().T.head(5))

def preprocess_dataset(df, data_type, target_col=None):
    df_clean = df.copy()
    num_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
    cat_cols = df_clean.select_dtypes(include=['object']).columns
    for col in cat_cols:
        if col != target_col:
            df_clean = pd.get_dummies(df_clean, columns=[col], drop_first=True)
    return df_clean

def select_features(X, y, k=5):
    k = min(k, X.shape[1])
    selector = SelectKBest(score_func=f_classif, k=k)
    X_new = selector.fit_transform(X, y)
    selected_cols = X.columns[selector.get_support()].tolist()
    return pd.DataFrame(X_new, columns=selected_cols), selected_cols

def split_dataset(X, y, test_size=0.2, val_size=0.1, random_state=42):
    X_tr_val, X_te, y_tr_val, y_te = train_test_split(X, y, test_size=test_size, random_state=random_state)
    val_ratio = val_size / (1.0 - test_size)
    X_tr, X_val, y_tr, y_val = train_test_split(X_tr_val, y_tr_val, test_size=val_ratio, random_state=random_state)
    return X_tr, X_val, X_te, y_tr, y_val, y_te

In [4]:
def run_experiment_1():
    print("="*60)
    print("=== LAUNCHING EXPERIMENT 1: AUTOMATED DATA PIPELINE ===")
    print("="*60)
    results = {}
    
    # Run on Iris
    iris_path = resolve_path("Datasets/iris/bezdekIris.data")
    if os.path.exists(iris_path):
        df_iris, dt = smart_loader(iris_path)
        print("\n[Iris Dataset EDA]")
        automated_eda(df_iris, dt)
        X = df_iris.drop(columns=['class'])
        y = df_iris['class']
        X_sel, sel_cols = select_features(X, y, k=3)
        X_tr, X_val, X_te, y_tr, y_val, y_te = split_dataset(X_sel, y)
        results['Iris'] = {'Samples': len(df_iris), 'Selected Features': sel_cols, 'Train Shape': X_tr.shape}
        
    # Run on Diabetes
    diab_path = resolve_path("Datasets/Diabetes_Dataset/diabetes.csv")
    if os.path.exists(diab_path):
        df_diab, dt = smart_loader(diab_path)
        print("\n[Diabetes Dataset EDA]")
        automated_eda(df_diab, dt)
        df_clean = preprocess_dataset(df_diab, dt, target_col='Outcome')
        X = df_clean.drop(columns=['Outcome'])
        y = df_clean['Outcome']
        X_sel, sel_cols = select_features(X, y, k=5)
        X_tr, X_val, X_te, y_tr, y_val, y_te = split_dataset(X_sel, y)
        results['Diabetes'] = {'Samples': len(df_diab), 'Selected Features': sel_cols, 'Train Shape': X_tr.shape}

    print("\n=== EXPERIMENT 1 PIPELINE COMPLETE ===")
    return results

In [5]:
# Master Execution Cell
ex1_output = run_experiment_1()
display(pd.DataFrame(ex1_output).T)

=== LAUNCHING EXPERIMENT 1: AUTOMATED DATA PIPELINE ===

[Iris Dataset EDA]
Dataset Shape: (150, 5)
Missing Values: 0
Summary Statistics:


,count,mean,std,min,25%,50%,75%,max
sepal_length,150.0,5.843333,0.828066,4.3,5.1,5.80,6.4,7.9
sepal_width,150.0,3.057333,0.435866,2.0,2.8,3.00,3.3,4.4
petal_length,150.0,3.758000,1.765298,1.0,1.6,4.35,5.1,6.9
petal_width,150.0,1.199333,0.762238,0.1,0.3,1.30,1.8,2.5



[Diabetes Dataset EDA]
Dataset Shape: (768, 9)
Missing Values: 0
Summary Statistics:


,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.845052,3.369578,0.0,1.0,3.0,6.00,17.0
Glucose,768.0,120.894531,31.972618,0.0,99.0,117.0,140.25,199.0
BloodPressure,768.0,69.105469,19.355807,0.0,62.0,72.0,80.00,122.0
SkinThickness,768.0,20.536458,15.952218,0.0,0.0,23.0,32.00,99.0
Insulin,768.0,79.799479,115.244002,0.0,0.0,30.5,127.25,846.0



=== EXPERIMENT 1 PIPELINE COMPLETE ===


,Samples,Selected Features,Train Shape
Iris,150,"[sepal_length, petal_length, petal_width]","(105, 3)"
Diabetes,768,"[Pregnancies, Glucose, BMI, DiabetesPedigreeFu...","(537, 5)"
